# Splitting and Embedding Text Using LangChain (Similarity Search)


In [ ]:
pip install -q -r ./requirements.txt

This notebook uses the versions of the libraries (OpenAI, LangChain, Pinecone, etc.) from requirements.txt.

Download [requirements.txt](https://drive.google.com/file/d/1XnmPWEdzrMeQHwfcsIOJyRqV8hChs-u1/view?usp=drive_link)

In [16]:
# pip install langchain-text-splitters

In [17]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

with open('files/churchill_speech.txt') as f:
    churchill_speech = f.read()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    length_function=len
)

In [8]:
chunks = text_splitter.create_documents([churchill_speech])
# print(chunks[2])
# print(chunks[10].page_content)
print(f'Now you have {len(chunks)}')

Now you have 300


#### Embedding Cost

In [9]:
def print_embedding_cost(texts):
    import tiktoken
    enc = tiktoken.encoding_for_model('text-embedding-3-small')
    total_tokens = sum([len(enc.encode(page.page_content)) for page in texts])
    # check prices here: https://openai.com/pricing
    print(f'Total Tokens: {total_tokens}')
    print(f'Embedding Cost in USD: {total_tokens / 1000 * 0.00002:.6f}')
    
print_embedding_cost(chunks)

Total Tokens: 4820
Embedding Cost in USD: 0.000096


### Creating embeddings

In [11]:
# import warnings
# warnings.filterwarnings('ignore', module='langchain')

In [10]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model='text-embedding-3-small', dimensions=1536)  # 512 works as well

In [11]:
vector = embeddings.embed_query(chunks[0].page_content)
vector

[0.021026611328125,
 0.04205322265625,
 0.079345703125,
 0.0191650390625,
 0.0002130270004272461,
 -0.0307464599609375,
 -0.0038967132568359375,
 -0.0015001296997070312,
 -0.0097503662109375,
 0.0180511474609375,
 0.0325927734375,
 -0.002593994140625,
 0.0169525146484375,
 -0.035400390625,
 -0.00458526611328125,
 0.0166473388671875,
 -0.01361846923828125,
 0.00394439697265625,
 0.0098114013671875,
 0.020904541015625,
 -0.02606201171875,
 -0.004550933837890625,
 -0.042724609375,
 -0.0188446044921875,
 -0.03411865234375,
 0.007724761962890625,
 -0.003448486328125,
 -0.047088623046875,
 0.02301025390625,
 -0.01290130615234375,
 0.0305938720703125,
 -0.0181121826171875,
 -0.0016450881958007812,
 0.03680419921875,
 0.027862548828125,
 -0.052947998046875,
 0.04443359375,
 -0.0132598876953125,
 -0.042572021484375,
 0.002040863037109375,
 -0.006664276123046875,
 -0.048614501953125,
 0.0299530029296875,
 0.04168701171875,
 0.01922607421875,
 -0.0670166015625,
 0.0146636962890625,
 -0.0250854492

### Inserting the Embeddings into a Pinecone Index

In [20]:
# Importing the libraries and initializing the Pinecone client
import os
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

pc = Pinecone()

In [ ]:
# deleting all indexes
indexes = pc.list_indexes().names()
for i in indexes:
    print(f'Deleting index {i} ... ', end='')
    pc.delete_index(i)
    print('Done')

Deleting all indexes ... Done
Deleting all indexes ... Done
Deleting all indexes ... Done


In [14]:
# creating an index
from pinecone import ServerlessSpec
index_name = 'churchill-speech'
if index_name not in pc.list_indexes().names():
    print(f'Creating index {index_name}')
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric='cosine',
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        ) 
    )
    print('Index created! 😊')
else:
    print(f'Index {index_name} already exists!')

Creating index churchill-speech
Index created! 😊


In [21]:
# processing the input documents, generating embeddings using the provided `OpenAIEmbeddings` instance,
# inserting the embeddings into the index and returning a new Pinecone vector store object. 
vector_store = PineconeVectorStore.from_documents(chunks, embeddings, index_name=index_name)

In [22]:
index = pc.Index(index_name)
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '186',
                                    'content-type': 'application/json',
                                    'date': 'Mon, 16 Mar 2026 10:45:03 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '38',
                                    'x-pinecone-request-latency-ms': '38',
                                    'x-pinecone-response-duration-ms': '40'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 300}},
 'storageFullness': 0.0,
 'total_vector_count': 300,
 'vector_type': 'dense'}

### Asking Questions (Similarity Search)

In [23]:
query = 'Where should we fight?'
result = vector_store.similarity_search(query)
print(result)

[Document(id='f651b0ec-7742-4019-986c-71d31ab34a8d', metadata={}, page_content='shall fight on the beaches, we shall fight on the landing grounds, we shall fight in the fields and'), Document(id='97e69ed4-0866-4130-9932-35a2f58905df', metadata={}, page_content='end, we shall fight in France, we shall fight on the seas and oceans, we shall fight with growing'), Document(id='76b627ed-8993-4835-8bcf-040c48f2a4de', metadata={}, page_content='streets, we shall fight in the hills; we shall never surrender, and even if, which I do not for a'), Document(id='aa413c93-26db-490b-91ab-3621eadc5c95', metadata={}, page_content='number of the enemy, and fought fiercely on some of the old grounds that so many of us knew so')]


In [24]:
for r in result:
    print(r.page_content)
    print('-' * 50)

shall fight on the beaches, we shall fight on the landing grounds, we shall fight in the fields and
--------------------------------------------------
end, we shall fight in France, we shall fight on the seas and oceans, we shall fight with growing
--------------------------------------------------
streets, we shall fight in the hills; we shall never surrender, and even if, which I do not for a
--------------------------------------------------
number of the enemy, and fought fiercely on some of the old grounds that so many of us knew so
--------------------------------------------------


### Answering in Natural Language using an LLM

In [27]:
from langchain_classic.chains import RetrievalQA
from langchain_openai import ChatOpenAI

# Initialize the LLM with the specified model and temperature
llm = ChatOpenAI(model='gpt-5.2', temperature=0.2)

# Use the provided vector store with similarity search and retrieve top 3 results
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 3})

# Create a RetrievalQA chain using the defined LLM, chain type 'stuff', and retriever
chain = RetrievalQA.from_chain_type(llm=llm, chain_type='stuff', retriever=retriever)

In [28]:
query = 'Answer only from the provided input. Where should we fight?'
answer = chain.invoke(query)
print(answer)

{'query': 'Answer only from the provided input. Where should we fight?', 'result': 'On the beaches; on the landing grounds; in the fields; in France; on the seas and oceans; in the streets; in the hills.'}


In [29]:
query = 'Who was the king of Belgium at that time?'
answer = chain.invoke(query)
print(answer)

{'query': 'Who was the king of Belgium at that time?', 'result': 'The king of Belgium at that time was **King Leopold** (referred to in the excerpt as “King Leopold”).'}


In [30]:
query = 'What about the French Armies??'
answer = chain.invoke(query)
print(answer)

{'query': 'What about the French Armies??', 'result': 'The context says the French Army was intended to advance across the Somme “in great strength to grasp it,” and that the area/objective was “held by the French troops.” It also notes that British and French armies fought. Beyond that, I don’t know—there isn’t more detail provided here about the French Armies.'}
